In [0]:
# from pyspark.sql import SparkSessionspark
# spark=SparkSessionspark.builder.appName("nishnatdesai's App").getOrCreate()
spark

In [0]:
dbutils.widgets.text("sass_tokeen","")
saskey=dbutils.widgets.get("sass_tokeen")
spark.conf.set(f"fs.azure.account.key.costlowstorage.dfs.core.windows.net", saskey)


In [0]:
from pyspark.sql.types import *
from pyspark.sql import Row
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max , row_number,lit,current_timestamp,date_format,to_date,broadcast
from pyspark.sql.types import IntegerType,StringType,DoubleType,FloatType,LongType,StructField,StructType,DateType,DecimalType,TimestampType
from pyspark.sql.functions import col, when,sha2,concat_ws
from delta.tables import DeltaTable
import json
import datetime
from decimal import Decimal


dbutils.widgets.text("table_name", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("raw_container", "")
dbutils.widgets.text("curated_container", "")
dbutils.widgets.text("watermark_value","")
dbutils.widgets.text("file_name","")

storage_account = dbutils.widgets.get("storage_account")
raw_container = dbutils.widgets.get("raw_container")
curated_container = dbutils.widgets.get("curated_container")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
run_id = dbutils.widgets.get("run_id")
watermark_value = dbutils.widgets.get("watermark_value")
file_name = dbutils.widgets.get("file_name")

# Set Delta configurations
spark.conf.set("spark.sql.adaptive.enabled","true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

spark.sql("""
          create table if not exists ingestion_log(
                table_name string,
                run_id string,
                load_type string,
                status string,
                raw_count long,
                processed_rows long,
                raw_path string,
                curated_path string,
                audit_path string,
                start_time timestamp,
                end_time timestamp,
                message string
          )
          using delta
          """)

log_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("raw_count", LongType(), True),
    StructField("processed_rows", LongType(), True),
    StructField("raw_path", StringType(), True),
    StructField("curated_path", StringType(), True),
    StructField("audit_path", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("message", StringType(), True)
])


def log_sql(status=None, message=None, raw_count=None,
            processed_rows=None, start_time=None, end_time=None,
            table_name=None, run_id=None, load_type=None,
            curated_path=None, audit_path=None, raw_path=None):

    log_data = [(
        table_name,
        run_id,
        load_type,
        status,
        int(raw_count) if raw_count is not None else None,
        int(processed_rows) if processed_rows is not None else None,
        raw_path,
        curated_path,
        audit_path,
        start_time,
        end_time,
        message
    )]

    log_df = spark.createDataFrame(log_data, log_schema)

    log_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("ingestion_log")


start_time = datetime.datetime.now()
status = "STARTED"
processed_rows = 0
error_message = None
max_modified_date = None
updated_count , inserted_count = 0,0
result = {}

raw_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/{table_name}/{file_name}"
curated_path = f"abfss://{curated_container}@{storage_account}.dfs.core.windows.net/{table_name}/fact_{table_name}/"
audit_path = f"abfss://{curated_container}@{storage_account}.dfs.core.windows.net/{table_name}/pipeline_audit/"

df_raw = spark.read\
    .format("parquet")\
        .load(raw_path)

df_raw = df_raw.withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))

from pyspark.sql import Row
from decimal import Decimal
import datetime

raw_count = df_raw.count()

log_sql(status=status,start_time=start_time,table_name=table_name,run_id=run_id,load_type=load_type,raw_count=raw_count,processed_rows=processed_rows,curated_path=curated_path,message=f"table load started",
            raw_path=raw_path,audit_path=audit_path)


# print("After adding test record:", df_raw.count())

if watermark_value in [None,"","NULL","null","None"]:
    watermark_value = None

if DeltaTable.isDeltaTable(spark, curated_path) and watermark_value is None:
    target_df = spark.read.format("delta").load(curated_path)
    watermark_value = (
        target_df.agg(spark_max("ModifiedDate").alias("max_modified_date"))
        .collect()[0]["max_modified_date"]
    )
else:
    watermark_value = watermark_value

# print("max_modified_date:", max_modified_date)

if load_type == "incremental" and watermark_value is not None:
    df_raw = df_raw.filter(
        col("ModifiedDate")> lit(watermark_value)
    )

df_transformed = (
df_raw
.dropDuplicates(["SalesOrderID"])
.withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
.withColumn("OrderDate", to_date("OrderDate"))
.withColumn("ShipDate", to_date("ShipDate"))
.withColumn("DueDate", to_date("DueDate"))
.withColumn("TotalDue", col("TotalDue").cast("double"))
.withColumn("order_year", col("OrderDate").substr(1,4).cast("int"))
.withColumn("order_month", col("OrderDate").substr(6,2).cast("int"))
.withColumn(
    "order_status_desc",
    when(col("Status") == 1, "In Process")
    .when(col("Status") == 2, "Approved")
    .when(col("Status") == 3, "Backordered")
    .when(col("Status") == 4, "Rejected")
    .when(col("Status") == 5, "Shipped")
    .when(col("Status") == 6, "Cancelled")
    .otherwise("Unknown")
)
.withColumn("processing_run_id", lit(run_id))
.withColumn("processing_timestamp", current_timestamp())
.withColumn("load_date", current_timestamp())
.withColumn("processed_file_name", lit(file_name))
.withColumn("effective_start", current_timestamp())
.withColumn("effective_end", lit(None).cast("timestamp"))
.withColumn("is_current", lit("Y"))
)


# suffle partition count
spark.conf.set("spark.sql.shuffle.partitions","100")

exclude_cols = [
    "effective_start",
    "effective_end",
    "is_current",
    "processing_timestamp",
    "load_date",
    "processing_run_id"
]

business_cols = [c for c in df_transformed.columns if c not in exclude_cols]

df_transformed = df_transformed.withColumn(
    "hash_value",
    sha2(
        concat_ws("||", *[col(c).cast("string") for c in business_cols]),
        256
    )
)

if DeltaTable.isDeltaTable(spark, curated_path):
    # AND target.order_year = source.order_year
    # AND target.order_month = source.order_month
    delta_table = DeltaTable.forPath(spark, curated_path)

    # Step 1: Expire changed records
    (
        delta_table.alias("target")
        .merge(
            df_transformed.alias("source"),
            """
            target.SalesOrderID = source.SalesOrderID
            AND target.is_current = 'Y'
            """
        )
        .whenMatchedUpdate(
            condition="target.hash_value <> source.hash_value",
            set={
                "is_current": "'N'",
                "effective_end": "current_timestamp()"
            }
        ).whenNotMatchedInsertAll()
        .execute()
        
    )

   
    active_target = spark.read.format("delta").load(curated_path) \
    .filter(col("is_current") == "Y") \
    .select("SalesOrderID", "hash_value")

    # Identify rows to insert (new or changed)
    df_to_insert = (
        df_transformed.alias("source")
        .join(
            active_target.alias("target"),
            col("source.SalesOrderID") == col("target.SalesOrderID"),
            "left"
        )
        .filter(
            col("target.SalesOrderID").isNull() |
            (col("source.hash_value") != col("target.hash_value"))
        )
        .select("source.*")
    )

    # Append new versions
    df_to_insert.write.format("delta").mode("append").save(curated_path)

    metric=delta_table.history(1) \
    .select("operationMetrics") \
    .collect()[0][0]
    inserted_count = int(metric.get("numTargetRowsInserted", 0))
    updated_count = int(metric.get("numTargetRowsUpdated", 0))
    processed_rows = inserted_count + updated_count
    
    status = "COMPLETED"

    log_sql(status=status,start_time=datetime.datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,message=f"load complted with inserted rows:{inserted_count} and updated rows:{updated_count}",raw_count=raw_count,end_time=datetime.datetime.now(),raw_path=raw_path,curated_path=curated_path)

    spark.sql(f"""
              OPTIMIZE delta.`{curated_path}` ZORDER BY (SalesOrderID)
              """)

else:
    # print("First load - creating Delta table")

    df_transformed.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("order_year", "order_month") \
        .save(curated_path)

    spark.sql(f"""
              ALTER TABLE delta.`{curated_path}` SET TBLPROPERTIES(
                  'delta.autoOptimize.optimizeWrite'='true',
                  'delta.autoOptimize.autoCompact'='true'
              )
              """)
    

result = {
    "processed_rows": int(processed_rows),
    "status": status,
    "table_name": table_name,
    "run_id": run_id,
    "max_modified_date": str(watermark_value),
}

dbutils.notebook.exit(json.dumps(result))
# print("=========== COMPLETED ===========")

In [0]:
# spark.read.format("delta").load(curated_path).filter(col('SalesOrderID')=='75097').orderBy(col('effective_start'), ascending=False).display()